# Emotional Sinhala Speech Dataset — Kaggle pilot

One-click pilot of `build_emotional_sinhala_dataset.py` on **one** SLBC radio-drama
episode (*Muwan Palassa*) from the Internet Archive.

**Manifest-first:** this notebook produces a `manifest.csv` + `report.json`, **not**
redistributed audio. The source items carry no license field — treat them as copyrighted.

**Before you run (Settings panel, right):**
1. **Accelerator → GPU T4 ×2**   2. **Internet → ON**
3. Add-ons → **Secrets** → add `HF_TOKEN` (your HuggingFace token).
4. On huggingface.co, accept the terms for **`pyannote/speaker-diarization-3.1`**
   (otherwise diarization degrades to a single `SPEAKER_UNK` and clips are not speaker-pure).


In [ ]:
# --- environment: pin one GPU + load HF token from the Kaggle secret ----------
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Kaggle gives T4 x2; pin to one GPU
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle secret.")
except Exception as e:
    print("No HF_TOKEN secret -> diarization will degrade to SPEAKER_UNK:", e)


In [ ]:
# --- get the pipeline code ----------------------------------------------------
# Option A (default): clone the project repo (make it public, or add a GitHub token).
import os
if not os.path.isdir("Dataset-creation-withEmotion"):
    !git clone https://github.com/DSEgrp18/Dataset-creation-withEmotion.git
%cd Dataset-creation-withEmotion
# Option B (offline / private repo): upload build_emotional_sinhala_dataset.py as a
# Kaggle Dataset, then copy it here instead of cloning:
#   !cp /kaggle/input/<your-dataset>/build_emotional_sinhala_dataset.py .


In [ ]:
# --- install deps -------------------------------------------------------------
# torch / torchaudio / numpy / scipy are PREINSTALLED on Kaggle with CUDA wheels.
# Do NOT reinstall them. Everything below is additive.
!pip install -q internetarchive librosa soundfile pyloudnorm faster-whisper \
    "pyannote.audio>=3.1" demucs transformers
print("deps installed")


In [ ]:
# --- stage 1 sanity: download only (no GPU needed, ~1-2 min) ------------------
!python build_emotional_sinhala_dataset.py --stage download --smoke \
    --identifiers muwan-palassa-140113 --work-dir /kaggle/working/eesd


In [ ]:
# --- full smoke: all 9 stages end-to-end on the first ~3 min (needs GPU) ------
# This is the wiring/quality check. Expect LOW yield on 3 min of multi-speaker drama.
!python build_emotional_sinhala_dataset.py --stage all --smoke \
    --identifiers muwan-palassa-140113 --work-dir /kaggle/working/eesd


In [ ]:
# --- inspect the yield funnel + emotion distribution --------------------------
import json
rep = json.load(open("/kaggle/working/eesd/report.json"))
print("FUNNEL (minutes):", json.dumps(rep.get("funnel_minutes", {}), indent=2))
print("USABLE YIELD %:", rep.get("yield_percent"))
print("COUNTS:", rep.get("counts", {}))

ed = rep.get("emotion_distribution", {})
cc = ed.get("category_counts", {})
if cc:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 3))
    plt.bar(list(cc.keys()), list(cc.values()))
    plt.title("Emotion distribution (usable clips)")
    plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()
else:
    print("No usable-clip emotion labels yet (tiny smoke sample often yields 0).")

import pandas as pd, os
for name in ("manifest.csv", "needs_manual_transcription.csv"):
    p = f"/kaggle/working/eesd/{name}"
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f"\n{name}: {len(df)} rows"); print(df.head())


## Review, then scale — do NOT auto-download the whole archive

The smoke run above only covers the first ~3 minutes with a tiny ASR model — it proves
the wiring, not the quality. To produce a real pilot on the **full episode**, drop `--smoke`
(defaults to `whisper large-v3`, ~10-20 min on T4):

```bash
!python build_emotional_sinhala_dataset.py --stage all \
    --identifiers muwan-palassa-140113 --work-dir /kaggle/working/eesd
```

Then **stop and review**:
- Read the yield funnel + emotion histogram above.
- Listen to a handful of `manifest.csv` clips and check their Sinhala text.
- Skim `needs_manual_transcription.csv` — these are the emotionally-salient clips that
  failed ASR (the ones you most want, kept for a human pass).

Only after that should you add more identifiers (e.g. the other Muwan Palassa episodes or
`RadioDramas-SLBC`). `/kaggle/working` is wiped between sessions — **download `manifest.csv`,
`needs_manual_transcription.csv`, and `report.json` before the session ends.**
